# Day 006 · 复利与贴现
**Compounding & Discounting** · 阶段 P1 · 量化基础

> 复利与贴现是金融最基础的数学。爱因斯坦把复利称作“第八大奇迹”,巴菲特一辈子财富的 89% 是 60 岁后赚到的——不是他变聪明,是前 60 年的复利在最后阶段集中显现。这节课讲懂复利公式 FV=PV×(1+r)^N、72 法则(5 秒心算翻倍年限)、净现值(NPV)、内部收益率(IRR)四个核心工具,以及税、通胀、手续费三大复利杀手。学完你能自己算房贷、年金保险、教育金、银行理财的真实回报,识破销售员“年化 4.5% 二十年翻一倍”的话术——大多数实际 IRR 只有 1.8-2.5%,远低于同期国债。**学会算 IRR,你就有了识破金融产品话术的 X 光机**。

---

**课件生成日期:** 2026-05-04  ·  **建议学习时长:** 16 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 推出复利公式 FV = PV × (1+r)^N,理解为什么时间是复利里第一变量
- 用 72 法则 5 秒钟心算"年化 r% 翻倍要多少年"
- 掌握贴现 = 复利的逆运算,理解 NPV 和 IRR 的物理意义,而不只是 Excel 公式
- 看懂年金保险/教育金/银行理财的"真实年化",识破"年化 4.5%"销售话术
- 量化通胀、税、手续费三大复利杀手对长期收益的吃损,以及为什么 Bogle 力推低费率指数基金

## 历史背景:爱因斯坦的"第八大奇迹"和一个真实的年金保险陷阱

相传 1955 年有人问爱因斯坦,人类最伟大的发明是什么。这位发明相对论的物理学家给的答案不是火、不是电、也不是核能,而是——复利。"懂复利的人就赚利息,不懂复利的人就付利息。"这句话此后成为投资圈引用最频繁的金句。

为什么爱因斯坦如此推崇?因为复利揭示了一个反直觉的事实:1.01 的 365 次方等于 37.78,1.02 的 365 次方等于 1377。每天进步百分之一,一年是 38 倍;每天进步百分之二,一年是 1377 倍。两倍的努力换 36 倍的回报——这就是非线性的暴力。巴菲特一辈子的财富里,百分之八十九是六十岁之后赚到的,这不是因为他六十岁后突然变聪明,而是因为前面六十年的复利在最后阶段才真正显现威力。

但复利也有阴影面。二零一零年代国内某大型保险公司热销的"分红型年金保险",销售员承诺"年化 4.5%、保本保收益、二十年翻一倍多"。听起来稳健是不是?可一旦把交保费的现金流序列拉出来,用 IRR 反算真实年化,大多数实际收益只有 1.8% 到 2.5%,远低于同期国债。差在哪?差在销售员只把"分红"部分算进去,没算手续费、没算前几年现金值低于已交保费的"隐性损失"。一个不会算 IRR 的家庭,就这样被困在二十年低收益合同里。

本节最重要的两句话:**复利是时间的朋友,但只忠于懂它的人**;**学会算 IRR,你就有了识破金融产品话术的 X 光机**。

**关键人物:**
- Albert Einstein(传说中的"复利第八大奇迹"格言来源)
- Warren Buffett(89% 的财富在 60 岁后赚到,复利的人形教科书)
- John Bogle(Vanguard 创始人,把"低费率 + 复利"做成全民武器)
- Robert Kiyosaki(《富爸爸穷爸爸》作者,把现金流和复利讲给普通家庭)

## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 复利公式 + 72 法则:5 秒心算翻倍年限

复利公式 FV = PV × (1+r)^N。本金 PV 在年化 r 的回报下,N 年后变成 FV。
例:1 万本金,年化 8%,30 年后 = 10000 × 1.08^30 ≈ 100627,约 10 倍。

72 法则:翻倍年限 ≈ 72 / r(用百分数,不用小数)。年化 6% → 12 年翻倍;年化 8% → 9 年;年化 12% → 6 年。这是给散户最重要的心算工具,记住一辈子用。

复利的非线性:50 年 vs 30 年的差距远大于 1.67 倍,因为前 30 年的本金本身在后 20 年也按复利滚动。这就是巴菲特从 60 岁到 89 岁这段时间财富翻了多少倍的根本原因——不是技术变了,是时间够长了。

```
FV = PV × (1+r)^N    ;    N_double ≈ 72 / r%
```

> **举例:** 一个 25 岁的人每月定投 2000 元,年化 8% 复利到 60 岁(35 年),账户余额 ≈ 458 万;同样金额从 35 岁开始定投到 60 岁(25 年),余额 ≈ 191 万。差 10 年差 267 万——复利不是数学,是时间。


### 2. 连续复利 vs 离散复利:e 在哪儿出现

同样名义年化 10%,如果按年复利,1 元一年后变 1.10;按月复利变 (1 + 0.10/12)^12 = 1.1047;按日复利变 1.1052;按连续复利变 e^0.10 = 1.1052。

连续复利 e^(rT) 是离散复利在分割频率趋于无穷时的极限。在量化里我们经常用连续复利,因为数学上更干净——加法可加,跟对数收益(D5 讲过)统一。

实操中:利息计算(银行存款、债券)用离散复利;期权定价、连续时间模型用连续复利。

```
FV_连续 = PV × e^(rT)    ;    R_log = ln(FV/PV) = rT
```

> **举例:** 你买一只名义年化 6% 的债券,如果是按月复利,12 个月真实收益 = (1+0.06/12)^12 - 1 ≈ 6.17%。差 17 个基点看起来小,放在 1000 万本金上一年差 1.7 万。


### 3. 贴现现金流(DCF)与净现值(NPV)

贴现是复利的逆运算。复利问"今天 1 万,N 年后多少",贴现问"N 年后的 1 万,今天值多少"。

折现率 r 是关键:r 越大,未来的钱越不值钱。给国债贴现用无风险利率(2-3%);给股票贴现要加风险溢价(8-12%);给比特币这种高波动资产贴现率更高(15-20%)。

NPV = 把一系列未来现金流全部贴现到今天再加总,减去初始投入。NPV > 0 项目值得做,NPV < 0 不值得。这是企业并购、房地产、教育金、年金险决策的核心数学。

```
PV = FV / (1+r)^N    ;    NPV = -C₀ + Σ_{t=1..N} CF_t / (1+r)^t
```

> **举例:** 你考虑一笔投资:今天投 10 万,未来 5 年每年回收 2.5 万。简单算总收 12.5 万,赚 2.5 万。但用 8% 折现率算 NPV ≈ -182 元,几乎为零。也就是说这笔交易真实回报率低于 8%,不如直接买理财。


### 4. IRR:把任何"看起来很美"的金融产品打回原形

IRR(内部收益率)是让 NPV = 0 的折现率,也就是这个项目本身能给你多少年化回报。Python 里 numpy_financial.irr(cashflows) 一行调用就出。

IRR 是散户面对销售话术最锐利的武器:任何人卖给你"分红型年金保险"、"教育金"、"5 年定期理财"、"养老储蓄计划",你只问一句"全周期现金流的 IRR 是多少",对方一定支支吾吾。因为大多数这类产品 IRR 只有 1.8-3%,跟同期国债比毫无优势。

IRR 也有盲区:多解(现金流多次符号变化时)、不能比较不同期限的项目(同 IRR 但 5 年和 30 年差别巨大)、隐含假设中间现金流按 IRR 再投资(实际不可能)。所以 IRR 是个尺子,不是终结性答案。

```
求解 r 使得 Σ CF_t / (1+r)^t = 0    ;    np_fin.irr(cashflows)
```

> **举例:** 某保险产品现金流:第 0-9 年每年交 5 万(支出),第 10-29 年每年返 3.5 万(收入)。算 IRR ≈ 2.7%。同期 10 年期国债收益率 3%。结论:这份保险还不如直接买国债,差距 0.3pp,30 年累计差约 25 万本金等价。


### 5. 复利的三大杀手:税、通胀、手续费

你看到的产品宣传"年化 8%",实际到手能有多少?三大杀手要扣:

1. **税**:股息分红 20% 个税(部分免税);券商佣金 0.03%、印花税卖出 0.1%;美股 ETF 分红 30% 预扣。这部分按交易频率每年吃损 0.3-1%。

2. **通胀**:中国近 10 年 CPI 平均 2.1%,但 M2 年化 8-10%。名义 8% 减真实通胀 3% = 实际 5%。

3. **手续费**:主动型公募基金平均管理费 1.5% + 申赎费 0.5%。看起来不大,30 年累计相当于本金的 1.7 倍消耗。Vanguard 创始人 John Bogle 名言:"在长期投资里,1% 的费率差,等于偷走 1/3 的最终财富。"

把这三项叠加,名义 8% 的真实购买力增长可能只有 3-4%。这就是为什么 Bogle 力推低费率指数基金——不是因为它能选出好公司,而是因为它把三大杀手中的一项压到了几乎零。

```
r_真实 ≈ r_名义 - 通胀 - 税率 - 费率
```

> **举例:** 主动管理基金近 10 年平均回报 7%(名义)、扣管理费 1.5%、扣分红税 0.6%、扣通胀 2.5% = 实际购买力增长 2.4%。同期沪深 300 指数 ETF 费率 0.15%,真实购买力增长约 5.7%。两者长期累积差距是巨大的。


## 实操:复利 + 贴现 + NPV + IRR 完整工具箱

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_006_compounding.py — 复利、72 法则、NPV、IRR 一次讲清
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt

P, r = 10000, 0.10
years = np.array([10, 20, 30, 40])
FV = P * (1 + r) ** years
print('=== 复利的时间魔力(本金 1 万,年化 10%) ===')
for y, v in zip(years, FV):
    print(f'  {y} 年: {v:,.0f} 元 ({v/P:.2f} 倍)')

rates = np.array([0.03, 0.05, 0.07, 0.10, 0.15])
rule72 = 72 / (rates * 100)
exact = np.log(2) / np.log(1 + rates)
print('\n=== 72 法则 vs 精确公式 ===')
for rate, est, exa in zip(rates, rule72, exact):
    print(f'  年化 {rate*100:.0f}%: 72法则 {est:.1f} 年 / 精确 {exa:.1f} 年 / 误差 {abs(est-exa):.2f} 年')

P, r_gross, T = 10000, 0.08, 30
fees = [0.0, 0.005, 0.01, 0.015, 0.02]
print('\n=== 三十年终值 vs 费率(本金 1 万,毛回报 8%) ===')
for f in fees:
    net_r = r_gross - f
    FV_net = P * (1 + net_r) ** T
    print(f'  费率 {f*100:.1f}%: 净 {net_r*100:.1f}% -> 30 年终值 {FV_net:,.0f} 元')

cash_flows = [-1_000_000] + [50_000] * 9 + [50_000 + 1_000_000]
print('\n=== 房产 NPV 在不同贴现率下 ===')
for r in [0.03, 0.05, 0.07, 0.10]:
    npv = sum(cf / (1 + r) ** t for t, cf in enumerate(cash_flows))
    label = '划算' if npv > 0 else '不划算'
    print(f'  贴现率 {r*100:.0f}%: NPV = {npv:,.0f} 元 ({label})')

irr = npf.irr(cash_flows)
print(f'\n=== 房产 IRR = {irr*100:.2f}% ===')

ts = np.arange(0, 41)
for rate in [0.03, 0.05, 0.07, 0.10, 0.15]:
    plt.plot(ts, P * (1 + rate) ** ts, label=f'{rate*100:.0f}% 年化')
plt.title('1 万本金的复利曲线 · 不同年化对比')
plt.xlabel('年数'); plt.ylabel('终值(元)'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('day006_compound.png', dpi=120)

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| A 股 | 沪深 300 指数 | 2005-2024 年化约 8.2%,加分红再投资约 9.5%。72 法则估翻倍约 9 年,实际翻倍 9 年——惊人吻合,作为长期复利锚点教学最直观。 |
| 美股 | SPY (S&P 500 ETF) | 1993-2024 年化约 10.3%,加分红再投资约 12.0%。72 法则翻倍约 7 年,实际约 6.5 年。费率仅 0.09%,是 Bogle 低费率哲学的典范。 |
| 中国债券 | 10 年期国债 | 2014-2024 收益率平均 3.5%,翻倍约 21 年。看起来低,但加上无信用风险,作为 NPV 折现率的基础非常稳健,任何"高收益、保本"产品都该跟它比。 |
| 不动产 | 房贷案例(等额本息) | 100 万房贷、30 年、年利率 5.0%,等额本息月供 5368,总还款 193 万。从银行视角看,银行的 IRR 就是 5.0%。从你的视角看,如果你的机会成本(可投资回报)> 5%,就该提前还款;< 5% 就慢慢还,差额拿去做更高回报的投资。 |


## 常见坑

### ⚠ 01. 用算术平均代替几何平均算长期年化

某基金 5 年收益 +50%、-30%、+40%、-25%、+20%。算术平均 = +11%,几何平均(真实复利)=(1.5×0.7×1.4×0.75×1.2)^(1/5) - 1 ≈ +5.0%。差 6 个百分点!业绩报表里的"年化"必须看清是几何还是算术。

### ⚠ 02. 把表面年化当真实年化

"分红型年金保险"宣传"年化 4.5%",但合同条款里有"前 5 年现金价值低于已交保费"、"分红不保证"、"退保扣手续费"等隐藏成本。把全部现金流拉出来算 IRR,常常只有 1.8-2.5%。凡是不肯给你完整现金流的产品,都是有水的。

### ⚠ 03. 折现率拍脑袋

不同风险的项目应该用不同的 r。给比特币现金流用 3% 折现率,会高估 NPV 数十倍。正确做法:无风险国债 r ≈ 3%,股票 r ≈ 8-10%,加密资产 r ≈ 15-20%,私募 PE 项目 r ≈ 12-15%。折现率选错,NPV 错得离谱。

### ⚠ 04. 忽视通胀,只看名义收益

"我买的理财产品年化 4%"——但同期 CPI 2%,真实购买力只增长 2%。过去 10 年 M2 年化 8-10%,你的真实购买力可能在贬值。所有长期理财必须扣通胀,看实际收益(real return)而不是名义收益(nominal return)。

### ⚠ 05. 主动基金 1.5% 管理费看起来无所谓

Bogle 算过:30 年期、年化 7%,管理费 0.1% vs 1.5%,最终财富相差 30%。这就是为什么美国先锋基金从 70 年代起死磕"低费率指数化"。中国市场早期主动基金费率 1.5-2.0% 普遍,近年指数 ETF 0.15% 已经普及,选对工具能省 1/3 财富。

## 实战 SOP · 复利与贴现实战 SOP

1. 任何一笔投资,先用 72 法则估翻倍年限。年化 < 4% 直接舍弃,年化 > 25% 警惕骗局
2. 任何"年化 X%"产品,都要看完整现金流序列,自己用 numpy_financial.irr 算 IRR
3. 折现率分层用:无风险国债 r=3%,股票项目 r=8-10%,加密资产 r=15-20%,看产品风险等级匹配
4. 长期投资优先选低费率工具:指数 ETF 费率 < 0.5%,主动基金费率 > 1.5% 直接谨慎
5. 看任何业绩报表,先确认是几何年化还是算术年化,差距大的(波动大)绝对要看几何
6. 通胀不能忽略:乘 0.6-0.7 折扣得真实购买力增长是常态,不是悲观
7. 30 年起规划:复利的暴力在最后 10 年才显现,前 20 年看不出威力,但要熬住
8. 凡是销售员不肯给你完整现金流序列、坚持"只看分红就好"的产品,直接走人

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 复利公式 FV = PV × (1+r)^N,72 法则 N_double ≈ 72/r%——散户终生用的心算工具
3. 连续复利 e^(rT) 是离散复利的极限,跟对数收益统一,量化里更常用
4. 贴现是复利的反向——把未来的钱按 r 折回今天才能跨期比较
5. NPV = 一系列贴现现金流之和,> 0 项目可做,< 0 不值得
6. IRR = 让 NPV = 0 的回报率,是识破金融产品话术的 X 光机
7. 名义回报 ≠ 真实购买力——税、通胀、手续费三大杀手平均吃 30-50% 收益
8. 巴菲特 89% 财富在 60 岁后获得——复利在最后 10-20 年才显现暴力
9. 1% 的费率差距,在 30 年累计上等于 1/3 财富(Bogle 名言),所以低费率指数基金是散户最强武器

## 自测题

**Q1.** 不查计算器,用 72 法则估算:年化 9% 的回报多少年翻倍?年化 12% 呢?

**Q2.** 你能用 60 秒讲清楚:为什么 IRR 是识破"分红型年金保险"陷阱的最锐利工具?

**Q3.** 同样年化 8% 的预期回报,管理费 1.5% 的主动基金 vs 管理费 0.15% 的指数 ETF,30 年后两者最终财富差距大约多少?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 007 · 风险与方差** (Risk as Variance)

Day 7:风险与方差——我们已经会算回报,接下来要会算风险。但金融里的风险不是"会不会亏",而是"收益散得多开"。下一节讲懂标准差、年化波动率、波动率聚集和下行半方差,以及索提诺比率为什么比夏普更适合非对称收益。

## 推荐阅读

- 《富爸爸,穷爸爸》Robert Kiyosaki——复利与现金流的入门启蒙(投资观点有争议,理财思维清晰)
- 《指数基金投资指南》John Bogle——1% 费率差的复杂故事 + 低费率指数化的数学论证
- 《Quantitative Investment Analysis》CFA Institute 教材 第 2 章——货币的时间价值、NPV、IRR 完整公式推导
- 巴菲特 1990 年致股东信——第二段讨论复利,经典必读(中文翻译可在雪球或巴菲特致股东信合集中找到)
- 《思考,快与慢》Daniel Kahneman 第 27 章——人类大脑无法直觉理解指数增长(为什么所有人都低估复利)